# 02 · Mandi Price Intelligence: EDA and Preprocessing

**Dataset:** Daily Commodity Prices India, yearly files in `data/raw/mandi/` (`2001.csv` … `2023.csv`).

**Output:** `data/cleaned/mandi/mandi_prices_monthly_cleaned.csv`, one row per State × Commodity × year × month, read by `src/models/train_mandi.py` and the Mandi Price page.

**Why monthly?** There are about 24 million daily rows (over 2 GB). Monthly averages keep the signal and make training practical. The raw files are read in chunks so that the whole dataset is never held in memory at once.

**Note:** if the raw CSV files are not in `data/raw/mandi/`, the cleaning cells are skipped and the notebook uses the existing cleaned monthly file for EDA only.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# Works whether Jupyter is opened from the project root or from the notebooks/ folder.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Project root:", ROOT)

## 1. Find the raw mandi files

In [ ]:
RAW_DIR = ROOT / "data" / "raw" / "mandi"
CLEAN_PATH = ROOT / "data" / "cleaned" / "mandi" / "mandi_prices_monthly_cleaned.csv"
RAW_FILES = sorted(RAW_DIR.glob("*.csv"))
RUN_RAW_CLEANING = len(RAW_FILES) > 0

print("Raw yearly files found:", len(RAW_FILES))
print("Will run raw cleaning:", RUN_RAW_CLEANING)
if not RUN_RAW_CLEANING:
    print("No raw mandi CSVs in data/raw/mandi/. Cleaning cells will be skipped; EDA uses the cleaned file.")

## 2. Inspect one raw file (first rows only)

Check the `Arrival_Date` format here. Dates such as `01/02/2015` can mean either January 2 or February 1, so the cleaning step depends on the format seen in this output.

In [ ]:
if RUN_RAW_CLEANING:
    sample = pd.read_csv(RAW_FILES[0], nrows=5, low_memory=False)
    print("Columns:", list(sample.columns))
    print(sample.dtypes)
    display(sample)
    print("Arrival_Date examples:", sample["Arrival_Date"].tolist())
else:
    print("Skipped (no raw files).")

## 3. Cleaning function (applied chunk by chunk)

Decisions:
- strip whitespace from text columns,
- convert `Modal_Price` to a number and drop rows where it is missing or not positive (a price of zero is treated as a recording error),
- drop rows with an unreadable `Arrival_Date`,
- aggregate each chunk to State × Commodity × year × month, keeping the observation count (`records`) so averages can be weighted correctly later.

In [ ]:
TEXT_COLS = ["State", "District", "Market", "Commodity", "Variety", "Grade"]

def clean_chunk(chunk):
    c = chunk.copy()
    for col in TEXT_COLS:
        if col in c.columns:
            c[col] = c[col].astype("string").str.strip()
    c["Modal_Price"] = pd.to_numeric(c["Modal_Price"], errors="coerce")
    c["Arrival_Date"] = pd.to_datetime(c["Arrival_Date"], errors="coerce", dayfirst=True)
    c = c[c["Modal_Price"].notna() & (c["Modal_Price"] > 0) & c["Arrival_Date"].notna()].copy()
    c["year"] = c["Arrival_Date"].dt.year
    c["month"] = c["Arrival_Date"].dt.month
    return (c.groupby(["State", "Commodity", "year", "month"])
             .agg(records=("Modal_Price", "size"), Modal_Price=("Modal_Price", "mean"))
             .reset_index())

## 4. Run cleaning over all raw files

Each chunk is aggregated first. The partial results are then combined with a **record-weighted** mean, so a month with more observations counts more. Without weighting, a thinly reported market would have the same influence as a busy one.

In [ ]:
if RUN_RAW_CLEANING:
    partials = []
    for path in RAW_FILES:
        print("Reading", path.name, flush=True)
        for chunk in pd.read_csv(path, chunksize=400_000, low_memory=False):
            partials.append(clean_chunk(chunk))
    p = pd.concat(partials, ignore_index=True)
    p["weighted_price"] = p["Modal_Price"] * p["records"]
    monthly = (p.groupby(["State", "Commodity", "year", "month"], as_index=False)
                 .agg(records=("records", "sum"), weighted_price=("weighted_price", "sum")))
    monthly["Modal_Price"] = monthly["weighted_price"] / monthly["records"]
    monthly = monthly[["State", "Commodity", "year", "month", "records", "Modal_Price"]]
    print("Monthly rows:", len(monthly), "| valid daily observations:", int(monthly["records"].sum()))
    CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
    monthly.to_csv(CLEAN_PATH, index=False)
    print("Saved cleaned dataset:", CLEAN_PATH)
else:
    print("Skipped: no raw files to clean.")

## 5. Validate the cleaned monthly dataset

In [ ]:
m = pd.read_csv(CLEAN_PATH)
print("Shape:", m.shape)
print("Years:", m["year"].min(), "to", m["year"].max())
print("States:", m["State"].nunique(), "| Commodities:", m["Commodity"].nunique())
print("\nMissing values:")
print(m.isna().sum())
print("\nDuplicate State-Commodity-month rows:", m.duplicated(["State", "Commodity", "year", "month"]).sum())
print("Non-positive prices:", (m["Modal_Price"] <= 0).sum())
display(m["Modal_Price"].describe().to_frame().T.round(2))

## 6. EDA: national yearly average price for the most-observed commodities

In [ ]:
top_commodities = (m.groupby("Commodity")["records"].sum().sort_values(ascending=False).head(5).index.tolist())
sub = m[m["Commodity"].isin(top_commodities)].copy()
sub["weighted_price"] = sub["Modal_Price"] * sub["records"]
yearly = sub.groupby(["Commodity", "year"], as_index=False).agg(
    weighted_price=("weighted_price", "sum"), records=("records", "sum"))
yearly["avg_price"] = yearly["weighted_price"] / yearly["records"]
display(yearly.head(10).round(2))

fig, ax = plt.subplots(figsize=(10, 5))
for name, g in yearly.groupby("Commodity"):
    ax.plot(g["year"], g["avg_price"], marker="o", label=name)
ax.set_title("Weighted average modal price by year (top 5 commodities by observation count)")
ax.set_ylabel("Rs per quintal")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()